# 🚗 Study Case 2 - Sauter Digital: Motor de Recomendação de Veículos (MVP)

**Objetivo de Negócio:** Melhorar a retenção de usuários apresentando alternativas viáveis de mercado no carrossel "Others you may like", resolvendo o problema de "Cold Start".

**Diretrizes do Produto:**
1. **Recomendação Baseada em Conteúdo:** Como não há histórico de navegação, usaremos as características técnicas dos veículos.
2. **Diversidade de Marcas:** O algoritmo deve priorizar ativamente a exibição de marcas concorrentes.
3. **Coerência Financeira:** Um carro de R$ 50k não deve ser recomendado para quem busca um de R$ 300k.

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURAÇÕES DO SISTEMA DE RECOMENDAÇÃO ---
CONFIG = {
    "MARGEM_PRECO": 0.25,          # Margem de ±25% para evitar discrepâncias irreais
    "TOP_N": 5,                    # Número de carros no carrossel
    "COLS_CAT": ['fuel', 'gear'],  # Texto para transformar em matemática
    "COLS_NUM": ['engine_size', 'age_years', 'avg_price_brl'], # Numéricas para escalonar
    "COL_MARCA": 'brand',          
    "COL_PRECO": 'avg_price_brl'   
}

### 1. Ingestão e Limpeza de Dados
[cite_start]Extração do dataset Kaggle e seleção das features relevantes, descartando colunas que não agregam valor à similaridade física do carro (como mês/ano de referência da tabela FIPE)[cite: 31, 47].

In [ ]:
# Carregando os dados brutos
# ATENÇÃO: Substitua pelo caminho real do dataset na sua máquina
caminho_dados = 'vagnerbessa_average_car_prices.csv' 

try:
    df_bruto = pd.read_csv(caminho_dados)
    
    # Selecionando as features que importam para o algoritmo
    features = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'age_years', 'avg_price_brl']
    df = df_bruto[features].copy()
    
    # Limpeza básica: removendo nulos e carros idênticos (deduplicação)
    df = df.dropna()
    df = df.drop_duplicates(subset=['brand', 'model', 'gear', 'fuel', 'engine_size', 'age_years'])
    df = df.reset_index(drop=True)
    
    print(f"Dataset pronto! Total de veículos únicos para análise: {df.shape[0]}")
except FileNotFoundError:
    print("Por favor, insira o caminho correto do arquivo CSV do Kaggle.")

### 2. Feature Engineering: Traduzindo Características em Matemática
[cite_start]Para que o algoritmo entenda a diferença entre características textuais e numéricas[cite: 48], aplicamos:
- **OneHotEncoder:** Transforma `fuel` e `gear` em matrizes binárias (0 ou 1).
- [cite_start]**RobustScaler:** Resolve o problema de escala massiva entre o tamanho do motor (`engine_size`) e o preço (`avg_price_brl`), sem ser distorcido por carros de altíssimo luxo (outliers)[cite: 49].

In [ ]:
# Criando o Pipeline de Transformação
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), CONFIG["COLS_NUM"]),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), CONFIG["COLS_CAT"])
    ])

# Gerando o "DNA Matemático" (Matriz) de cada carro
matriz_ml = preprocessor.fit_transform(df)
print(f"Formato da matriz matemática: {matriz_ml.shape}")